# 레이아웃 만들기

## 0) 기본 설정 (와이드 모드)

In [ ]:
import streamlit as st
st.set_page_config(page_title="레이아웃 데모", layout="wide")

#### -layout="wide": 더 넓은 화면에서 대시보드형 UI 구성에 유리

## 1) 사이드바(sidebar)와 메인 영역

In [ ]:
with st.sidebar:
    st.header("필터")
    date = st.date_input("날짜")
    cls = st.selectbox("클래스", ["전체","A","B"])

st.title("대시보드")
st.write("사이드바에서 필터를 조정하세요.")

#### 패턴: 입력(필터)은 sidebar, 결과는 메인에 표시 → 사용성이 좋음

## 2) 컬럼(columns)로 가로 배치

In [ ]:
col1, col2, col3 = st.columns([2, 3, 2])  # 비율로 너비 제어
with col1:
    st.subheader("요약 지표")
    st.metric("Accuracy", "93.2%", "+0.7%")
with col2:
    st.subheader("라인 차트")
    st.line_chart({"acc":[0.8,0.85,0.9,0.932]})
with col3:
    st.subheader("세부 옵션")
    st.checkbox("스무딩")


#### 꿀팁: 리스트로 비율 지정 → 반응형 + 균형 잡힌 배치

## 3) 탭(tabs)으로 화면 전환

In [ ]:
tab1, tab2, tab3 = st.tabs(["개요", "지표", "로그"])
with tab1:
    st.write("한 눈에 보는 개요")
with tab2:
    st.write("정밀 지표 표/차트")
with tab3:
    st.code("학습 로그 미리보기...")

#### 패턴: “개요/지표/원본데이터(로그)” 3단 탭은 수업에서 아주 인기

## 4) 확장(expander)로 추가 정보 접기/펼치기

In [ ]:
with st.expander("전처리 설명 보기"):
    st.markdown("- 이진화 → 자르기 → 패딩 → 28x28 리사이즈")

#### 본문을 깔끔하게 유지하면서 부가 설명 제공

## 5) 컨테이너(container)와 플레이스홀더(placeholder)

In [ ]:
placeholder = st.empty()        # 화면의 빈 자리 확보
with st.container():
    st.write("섹션 시작")
    st.write("여러 컴포넌트를 묶어 관리")
# 동적으로 업데이트
placeholder.metric("현재 단계", "로딩 중...")
# ... 처리 후
placeholder.metric("현재 단계", "완료")

#### 패턴: 로딩/스트리밍 진행 상태 표시, 실시간 업데이트 UI

## 6) 폼(form)으로 입력 묶기 (Submit까지 한 번에)

In [ ]:
with st.form("hyperparams"):
    lr = st.number_input("Learning Rate", 0.0001, 1.0, 0.001, format="%.4f")
    epochs = st.slider("Epochs", 1, 200, 30)
    submitted = st.form_submit_button("학습 시작")
if submitted:
    st.success(f"LR={lr}, Epochs={epochs}로 학습 시작!")


#### 장점: 여러 입력을 한 번에 제출 → 예측/학습 파라미터 설정에 딱

## 7) 세션 상태(session_state)로 상호작용 기억

In [ ]:
if "filters" not in st.session_state:
    st.session_state.filters = {"cls":"전체"}

st.session_state.filters["cls"] = st.sidebar.selectbox(
    "클래스", ["전체","A","B"], index=["전체","A","B"].index(st.session_state.filters["cls"])
)

st.write("선택된 클래스:", st.session_state.filters["cls"])

#### 활용: 탭 전환/재실행에도 선택값 유지
#### 스트림릿(streamlit)의 st.session_state는 웹 앱이 매번 리렌더링(스크립트 재실행) 될 때도 사용자의 상태(state)를 기억하는 저장소입니다.

# 왜 필요한가?

## - **기본 동작**: Streamlit은 사용자가 버튼 클릭, 슬라이더 조정 등 이벤트를 발생시키면 전체 
## 스크립트를 위에서부터 다시 실행합니다.
## - 이때 일반 변수는 초기화되므로 값이 유지되지 않습니다.
## - → `st.session_state`를 사용하면 **사용자 인터랙션, 파라미터, 상태**를 보존할 수 있습니다.

# 주요 특징

## 딕셔너리 형태 : st.session_state는 파이썬 딕셔너리처럼 동작합니다.

In [ ]:
st.session_state["count"] = 1
print(st.session_state["count"])

## 위젯과 자동 연동 : 위젯에 key를 지정하면 그 값이 자동으로 session_state에 저장됩니다.

In [ ]:
name = st.text_input("이름을 입력하세요", key="username")
st.write("session_state:", st.session_state.username)

## 3. **앱 실행 동안 유지**
###    - 사용자가 새로고침하기 전까지 같은 세션에서 값이 유지됩니다.
###    - 다른 사용자는 각자의 `session_state`를 가집니다.
## 4. **동적 상태 관리**
###    - 버튼 클릭 횟수, 체크박스 상태, 모델 예측 결과 저장 등 **상태 기반 UI** 구현 가능

# 간단 예제

In [ ]:
import streamlit as st

# 초기화
if "counter" not in st.session_state:
    st.session_state.counter = 0

# 버튼 클릭 시 상태값 변경
if st.button("증가"):
    st.session_state.counter += 1
if st.button("감소"):
    st.session_state.counter -= 1

st.write("현재 값:", st.session_state.counter)

#### 버튼을 눌러도 앱은 전체 리렌더링 되지만, counter 값은 session_state 덕분에 유지됩니다.

# 활용 패턴

### - **로그인 상태 유지** (사용자 ID/토큰 저장)
### - **탭 간 데이터 공유** (탭에서 전처리한 결과를 다른 탭에서 사용)
### - **다단계 워크플로우** (Step1 입력 → Step2 처리 → Step3 출력)
### - **버튼 토글/체크박스 기억하기**
### - **모델/데이터 캐시**와 조합하여 효율적 실행

# 8) 공통 대시보드 레이아웃 템플릿 (복붙용)

In [ ]:
import streamlit as st
st.set_page_config(page_title="모델 대시보드", layout="wide")

# --- Sidebar ---
with st.sidebar:
    st.header("필터")
    dataset = st.selectbox("데이터셋", ["Val","Test"])
    smooth = st.slider("스무딩", 1, 25, 5)
    show_points = st.checkbox("포인트 표시", False)

# --- Header ---
st.title("모델 성능 대시보드")
st.caption("실험 비교 · 지표 요약 · 예측 분포")

# --- KPI Row ---
k1, k2, k3, k4 = st.columns(4)
k1.metric("Best Val Acc", "93.2%")
k2.metric("Min Val Loss", "0.183")
k3.metric("Latency(ms)", "12.4")
k4.metric("Params(M)", "21.8")

# --- Tabs ---
t1, t2, t3 = st.tabs(["학습 곡선", "지표표/혼동행렬", "예측 샘플"])

with t1:
    c1, c2 = st.columns([3,2])
    with c1:
        st.subheader("Loss Curve")
        st.line_chart({"train":[0.9,0.6,0.4,0.25], "val":[1.0,0.7,0.5,0.3]})
    with c2:
        st.subheader("Accuracy Curve")
        st.line_chart({"train":[0.5,0.7,0.85,0.92], "val":[0.45,0.65,0.8,0.9]})

with t2:
    st.subheader("지표 테이블")
    st.table({"precision":[0.91,0.88], "recall":[0.90,0.86], "f1":[0.905,0.87]})
    with st.expander("혼동행렬 보기"):
        st.dataframe({"Pred 0":[88,5], "Pred 1":[7,100]})

with t3:
    left, right = st.columns([2,3])
    with left:
        st.subheader("입력 샘플")
        st.image("https://placehold.co/300x300", caption="업로드/샘플")
    with right:
        st.subheader("Top-k 확률")
        st.bar_chart({"A":[0.7], "B":[0.2], "C":[0.1]})


## 9) 레이아웃 설계 베스트 프랙티스

#### - **정보 구조 먼저**: “필터 → KPI(요약) → 상세 탭(곡선/표/원본)” 순으로
#### - **한 화면 = 한 목적**: 탭으로 역할 분리(개요/지표/원본/로그)
#### - **시각적 균형**: `columns([3,2])`처럼 비율로 그리드 잡기
#### - **설명은 expander**에 넣어 본문 간결화
#### - **성능 고려**: 무거운 계산은 `st.cache_data/resource`로 캐시
#### - **재활용**: 템플릿(위 8번)에서 컴포넌트만 갈아끼기